## Library and Imports

In [0]:
# Import statements 
from collections import Counter
import xlrd
import pandas as pd
import numpy as np
import time
import os
import io
import re
import json
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.functions import sum as spark_sum, col, monotonically_increasing_id,row_number
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.window import Window

In [0]:
exec(open('/Workspace/Shared/Library.py').read())
with open('credentials.json', 'r') as f:
    credentials = json.load(f)
connection_string = credentials.get('connection_string')
with open('mapping.json','r') as f:
    mapping_config = json.load(f)

## Loading data from Azure

In [0]:
# Connect to azure blobs container to obtain input files
container = "temp-test"
CAD_USD_FX = 0.7
GBP_USD_FX = 1.32
input_filename = 'campaign_metadata.csv','google_ads_daily.json','meta_ads_daily.csv','store_visits.csv'
get_azure_container_file(connection_string,container,input_filename)

files = [
    ('campaign_metadata.csv', 'CAMPAIGN_METADATA'),
    ('meta_ads_daily.csv', 'META_ADS_DAILY'),
    ('google_ads_daily.json', 'GOOGLE_ADS_DAILY'),
    ('store_visits.csv', 'STORE_VISITS')
]

spark.conf.set("spark.sql.session.timeZone", "UTC")

## Function definitions

In [0]:
def formalize_column_name(column_name: str) -> str:
    column_name = column_name.strip().lower()
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    return re.sub(r"_+", "_", column_name).strip("_")

def rename_columns(df: DataFrame,mapping_config: dict,mapping_name: str) -> DataFrame:
    mapping_name = mapping_name.split('.')[0]
    if mapping_name not in mapping_config:
        raise ValueError(f"Mapping '{mapping_name}' not found.")
    config = mapping_config[mapping_name]
    mapping = config["mapping"]

    column_mapping = (
        mapping.get("dimensions", [])
        + mapping.get("measurements", [])
    )
    
    normalized_mapping = {
        formalize_column_name(source): target
        for source, target in column_mapping
    }

    # Rename columns while preserving unmapped columns
    for old_name in df.columns:
        normalized_name = formalize_column_name(old_name)

        if normalized_name in normalized_mapping:
            new_name = normalized_mapping[normalized_name]

            if old_name != new_name:
                df = df.withColumnRenamed(old_name, new_name)

    return df

def correct_timezone(df: DataFrame,config: dict,time_column: str = "Time") -> DataFrame:
    timezone = config.get("timezone")

    if not timezone or time_column not in df.columns:
        return df

    value = F.trim(F.col(time_column))

    # Detect ISO timestamps with an explicit timezone:
    # Z, +08:00, -07:00, etc.
    has_timezone = value.rlike(r"(?i)(Z|[+-]\d{2}:\d{2})$")
    # Parse timestamps with an explicit timezone offset.
    utc_timestamp = F.to_utc_timestamp(
        F.to_timestamp(value, "yyyy-MM-dd'T'HH:mm:ssXXX"),
        "UTC"
    )
    # Parse timezone-naive timestamps in their configured timezone.
    if timezone:
        local_timestamp = F.to_utc_timestamp(
            F.to_timestamp(value, "yyyy-MM-dd HH:mm:ss"),
            timezone
        )
    else:
        local_timestamp = F.lit(None).cast("timestamp")

    # Explicitly zoned values are already absolute instants.
    # Naive values require a configured source timezone.
    result = (
        F.when(has_timezone, utc_timestamp)
        .otherwise(local_timestamp)
    )
    return df.withColumn(
        time_column,
        F.date_format(result, "yyyy-MM-dd HH:mm:ss")
    )
def fx_conversion(df,cad_fx,gbp_fx):
    if "Currency" not in df.columns:
        return df
    df_fx = df.filter((col("Currency").isNotNull()) & (col("Currency") != "USD"))
    # Convert Conversion_spend_usd and Spend to USD if 'Currency' is CAD or GBP
    if "Currency" in df_fx.columns and ("Conversion_spend_usd" in df_fx.columns or "Spend" in df_fx.columns):
        fx_case = (
            F.when(col("Currency") == "CAD", F.lit(cad_fx))
                .when(col("Currency") == "GBP", F.lit(gbp_fx))
                .otherwise(F.lit(1.0))
        )
        if "Conversion_spend_usd" in df.columns:
            df_fx = df_fx.withColumn(
                "Conversion_spend_usd",
                col("Conversion_spend_usd") * fx_case
            )
        if "Spend" in df_fx.columns:
            df_fx = df_fx.withColumn(
                "Spend",
                col("Spend") * fx_case
            )
        # After FX change, set currency code to USD
        if "Currency" in df_fx.columns:
            df_fx = df_fx.withColumn("Currency", F.lit("USD"))
    df = df.filter((col("Currency") == 'USD') | (col("Currency").isNull()))
    df = merge_dataframes(df, df_fx)
    return df

def standalize(df: DataFrame, mapping_config: dict, filename: str) -> DataFrame:
    mapping_name = filename.split('.')[0]
    config = mapping_config[mapping_name]

    if "measurements" in config["mapping"]:
        measure_cols = [formalize_column_name(source) for source, _ in config["mapping"]["measurements"]]
        df = df.fillna(0, subset=measure_cols)

    ## rename columns based on mapping files
    df = rename_columns(df, mapping_config, filename.lower())

    ## Correct timezone listed in Config["timezone"] to UTC for all time columns
    df = correct_timezone(df, config)
    
    ## Deduplication for AD data 
    if config["type"] == 'ad':
        if len(df.columns) > 0:
            mapping_keys_list = config["mapping_key"]
            df = fx_conversion(df,CAD_USD_FX,GBP_USD_FX)
            df_keys = [key for key in mapping_keys_list if key in df.columns]
            # print("before deduplication: ",df.count())
            df = df.withColumn("order_column", monotonically_increasing_id())
            window_spec = Window.partitionBy(*df_keys).orderBy(col("order_column").desc())
            df = df.withColumn("row_num", row_number().over(window_spec))
            df = df.filter(col("row_num") == 1).drop("row_num", "order_column")
            # print("after deduplication: ",df.count())
            
    ## Convert Google micro spend to Spend
    if 'google' in filename.lower():
        df = df.withColumn("Spend", col("Spend_micro") / 1000000)
        df = df.withColumn("Conversions_spend", col("Conversions_spend_micro") / 1000000)
        df = df.drop("Spend_micro").drop("Conversions_spend_micro")
    return df 

def createEmptySparkDf():
    schema = StructType([StructField('CODE', StringType())])
    df = spark.createDataFrame([(None,)], schema)
    empty_df = df.drop('CODE')
    return(empty_df)

def merge_dataframes(df1, df2):
    missing_cols = set(df2.columns) - set(df1.columns)
    for col in missing_cols:
        df1 = df1.withColumn(col, lit(None))
    missing_cols = set(df1.columns) - set(df2.columns)
    for col in missing_cols:
        df2 = df2.withColumn(col, lit(None))
    merged_df = df1.unionByName(df2)
    return merged_df
    
def data_staging(files,mapping_config, mode = "overwrite"):
    for filename, table_name in files:
        if filename.endswith('.csv'):
            df = pd.read_csv(filename)
        elif filename.endswith('.json'):
            df = pd.read_json(filename, lines=True)
            df = pd.json_normalize(df.to_dict(orient="records"), sep="_")
        else:
            continue
        print("Start processing: ",filename)
        df = spark.createDataFrame(df)
        df = standalize(df,mapping_config,filename)
        print("Writing total rows: ",df.count())
        df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"hive_metastore.staging.{table_name}")
        print(f"Table {table_name} loaded to staging")
    return


## Main

In [0]:
def main():
  time1 = time.time()
  data_staging(files,mapping_config)
  time2 = time.time()
  print("Load to Staging: {:.1f} seconds".format(time2 - time1))

In [0]:
main()

Start processing:  campaign_metadata.csv
Writing total rows:  24
Table CAMPAIGN_METADATA loaded to staging
Start processing:  meta_ads_daily.csv
Writing total rows:  3603
Table META_ADS_DAILY loaded to staging
Start processing:  google_ads_daily.json
Writing total rows:  1028
Table GOOGLE_ADS_DAILY loaded to staging
Start processing:  store_visits.csv
Writing total rows:  2300
Table STORE_VISITS loaded to staging
Load to Staging: 347.1 seconds


## [DNU] Test and debug cell only

## Cleanup files

In [0]:

for file_name in input_filename:
    file_path = file_name
    if (file_path.endswith('.csv') or file_path.endswith('.xlsx') or file_path.endswith('.json') or file_path.endswith('.zip')):
        os.remove(file_path)
        print(f"Deleted: {file_path}")
    else:
        print(f"Not a file: {file_path}")

Deleted: campaign_metadata.csv
Deleted: google_ads_daily.json
Deleted: meta_ads_daily.csv
Deleted: store_visits.csv
